# 3D Fluorescence LIVE/DEAD Segmentation

This notebook is a **generic template** for 3D fluorescence image segmentation and quantification of LIVE/DEAD-stained samples. All channels are merged and segmented together — no dedicated NUCLEI channel is needed.

**Before running the workflow:**
1. Replace the example file path, ROI, experiment name, and channel definitions in **Section 2** with your own data.
2. Make sure the channel names in `stain_dict` (Cell 5) match the metadata printed by Cell 4.
3. Start with the default processing settings and tune only the parameters that need adjusting for your dataset.

**Segmentation methods available:**
- **Watershed** (default) — classical multi-scale erosion + EDT peaks. Tunable via `nuclei_split_config`.
- **Cellpose 3D** — deep-learning model; handles touching cells natively. Set `trig_cellpose = True`.
- **StarDist** — 2D-to-3D translation model. Set `trig_stardist = True`.

---

# 1. Import Libraries and Helpers

Run the two cells below to load all required Python packages and the helper functions from `helpers/notebook_helpers.py`.

- **Cell 1** imports the full library set (napari, scipy, skimage, etc.).
- **Cell 2** reloads the helper module. Re-run it any time you edit the helper file without restarting the kernel.

In [ ]:
#[Cell 1]
# Import all required libraries for image processing and visualization.
from helpers.notebook_setup_helpers import load_ld_notebook_setup
globals().update(load_ld_notebook_setup())

In [ ]:
#[Cell 2]
# Reload helper functions from the external module.
# Rerun this cell after editing helpers/notebook_helpers.py.
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

Edit the values in the following cells to match **your own dataset** before running the analysis.

- **Cell 3** — all user-configurable parameters: file path, ROI, diameters, scaling, segmentation method, and split settings. This is the only cell you *must* edit for each new dataset.
- **Cell 4** — loads the image, reads voxel sizes and file metadata, and computes derived parameters. Whether the file is read all at once or lazily in chunks is decided automatically from its size — no setting needed, but the choice is printed as "Lazy (chunked) loading used". Review the printed output to verify the file was loaded correctly.
- **Cell 5** — defines the LIVE/DEAD channel mapping (`stain_dict`). Do **not** add a NUCLEI entry; all channels are merged for segmentation. Channel names must match the metadata printed by Cell 4.

In [ ]:
#[Cell 3]

# ---------------------------------------------------------------------------
# 1. Input file & ROI
# ---------------------------------------------------------------------------
# input_file: your microscopy file. Supported examples include `.nd2` and
# multi-channel `.tif/.tiff` files.
input_file = 'path/to/your_image_file.nd2'

# ROI: [x0, x1, y0, y1, z0, z1]. Use 0 for any limit you want to keep
# unchanged. Example: ROI = [100, 900, 150, 1200, 0, 40]
# Note: how the file is read (all at once, or lazily in chunks) is decided
# automatically by Cell 4 based on the file's size — no setting needed here.
ROI = [0, 0, 0, 0, 0, 0]

# ---------------------------------------------------------------------------
# 2. Experiment name & reused settings
# ---------------------------------------------------------------------------
# name_setup: prefix used when saving setup files and output tables.
name_setup = 'your_experiment_name'

# use_setup: True = reuse previously saved napari contrast/gamma settings if
# the matching CSV file exists.
use_setup = True

# automatic_contrast: True = when Cell 7 would otherwise open a napari window
# for interactive contrast/gamma adjustment (i.e. no reusable setup CSV —
# see use_setup above), skip it and instead pick Cont_min/Cont_max per
# channel automatically from the histogram (one above the histogram peak, up
# to the 99th percentile intensity to avoid saturation); Gamma stays at 1.0.
#   False (default): unchanged, interactive napari viewer.
automatic_contrast = False

# ---------------------------------------------------------------------------
# 3. Object sizes & image scaling
# ---------------------------------------------------------------------------
# Approximate diameters used during segmentation (in micrometers).
nuclei_diameter = 10  # um, approximate cell diameter for LIVE/DEAD-stained cells
cell_diameter = 30    # um, used only for threshold window sizing

# Keep these at 1.0 unless you intentionally want an extra manual scale
# correction.
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]

# ---------------------------------------------------------------------------
# 4. Segmentation method
# ---------------------------------------------------------------------------
# Enable ONE of the following, or leave both False for watershed.
#   - trig_cellpose: Cellpose 3D deep-learning model (recommended, requires
#     GPU). Segments directly from the intensity image; handles touching
#     cells natively.
#   - trig_stardist: StarDist 2D-to-3D translation model.
trig_cellpose = False
trig_stardist = False

# ---------------------------------------------------------------------------
# 5. Quantification
# ---------------------------------------------------------------------------
# multilabel: True = compute multi-marker combinations in the quantification
# summary.
multilabel = True

# ---------------------------------------------------------------------------
# 6. Cell split configuration
# ---------------------------------------------------------------------------
# Controls how aggressively watershed pulls apart touching cells that would
# otherwise be left merged as one blob (used even for the LIVE/DEAD union
# segmentation). Applies to the WATERSHED method only (skipped if
# trig_cellpose/trig_stardist are used). See get_nuclei_split_config() for
# the "conservative"/"balanced"/"aggressive" profile options.
nuclei_split_config = get_nuclei_split_config(profile="conservative")

# z_split_aggressive: controls how z-stack consistency is enforced after
# watershed.
#   - False (default): topology-only — splits only truly disconnected 3D
#     regions.
#   - True: watershed-reinforced — splits when any z-slice has multiple 2D
#     components.
nuclei_split_config["z_split_aggressive"] = False

In [ ]:
#[Cell 4]
# Load image, read metadata, and compute derived parameters. Whether the
# file is read lazily (chunked) or all at once is decided automatically
# from its size (printed below as "Lazy (chunked) loading used").
(meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
 cyto_factor, PCM_factor, zoom_factors, used_lazy_loading) = initialize_dataset(
    input_file, ROI,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    scale_factor=scale_factor,
    zoom_factors=zoom_factors,
)

In [ ]:
#[Cell 5]
# ---------------------------------------------------------------------------
# LIVE/DEAD channel mapping
# ---------------------------------------------------------------------------
# Define the channels in your dataset.
# Format:
#   'CONDITION_NAME': ['Marker label', 'channel name from file metadata', 'display color']
# Note: do NOT add a NUCLEI entry for LIVE/DEAD assays; all channels are
# merged for segmentation.
stain_dict = {
    'LIVE': ['CALCEIN', 'CHANNEL_LIVE', 'Green'],
    'DEAD': ['ETHD', 'CHANNEL_DEAD', 'Red'],
}

# 3. Preparation and Preview

This section builds the image stack, creates the staining table, and opens a napari viewer so you can inspect each channel and set contrast/gamma values.

- **Cell 6** — prepares the image stack (axes reordering, float conversion), builds `stain_df` from your `stain_dict`, and opens a napari viewer showing each channel.
- **Cell 7** — loads or creates per-channel contrast and gamma settings. If a saved setup CSV exists (from a previous run with the same `name_setup`), it is reused automatically; otherwise a napari viewer opens for interactive adjustment — unless `automatic_contrast = True` (set in Cell 3), in which case the viewer is skipped and `Cont_min`/`Cont_max` are picked automatically per channel from the histogram (Gamma stays at 1.0).
- **Cell 8** — displays the complete stain settings table. Review the contrast limits and gamma values before proceeding.

In [ ]:
#[Cell 6]
# Prepare image stack, build stain table, and open napari channel preview.
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, progress=tqdm,
)

In [ ]:
#[Cell 7]
# Load or create napari contrast/gamma settings for each channel.
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    automatic_contrast=automatic_contrast,
    settings=settings,
    napari_module=napari,
    progress=tqdm,
)

In [ ]:
#[Cell 8]
# Display the complete stain settings table for review.
stain_complete_df

# 4. Image Processing and Segmentation

This section applies the full image processing pipeline and then segments the structures of interest. Run the cells sequentially; each stage feeds into the next.

**Processing pipeline (Cells 9-15):**
- Normalization, isotropic resampling, median denoising, contrast/gamma adjustment, Gaussian smoothing, histogram equalization, and thresholding. Each step displays a histogram so you can monitor the effect.
- Cells 13-15 contain **user-tunable parameters** (`sigma`, `num_plateaus`, `plateau_factor`, `threshold_method`). Adjust them if the default values do not suit your data.

**Histogram export (Cell 16):**
- Saves histograms for every processing stage and a Parameters sheet to an Excel file.

**Segmentation (Cells 17-19):**
- Cell 17 segments cells from the combined LIVE+DEAD binary mask using the method selected in Cell 3. A virtual NUCLEI entry is added so downstream helpers work correctly.
- Cell 18 assigns the segmented labels to each LIVE/DEAD channel.
- Cell 19 opens napari viewers showing all processing stages and segmentation results side by side.

In [ ]:
#[Cell 9]
# Normalize each channel independently to [0, 255].
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

In [ ]:
#[Cell 10]
# Resample the image to isotropic voxel spacing.
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zoom_factors,
    meta=meta,
)

In [ ]:
#[Cell 11]
# Apply median filter for noise removal.
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

In [ ]:
#[Cell 12]
# Apply per-channel contrast and gamma adjustments from stain settings.
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

In [ ]:
#[Cell 13]
# ---------------------------------------------------------------------------
# Gaussian smoothing
# ---------------------------------------------------------------------------
# sigma: Gaussian kernel width in voxels.
#   0 = no smoothing · 0.5 = mild (default) · 1–2 = stronger (noisy data).
sigma = 0.5

# Apply Gaussian smoothing.
im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=sigma,
)

In [ ]:
#[Cell 14]
# ---------------------------------------------------------------------------
# Histogram equalization
# ---------------------------------------------------------------------------
# num_plateaus: clipping plateaus for histogram equalization.
#   2 (default) suits most data · 3–4 for low-contrast or dim channels.
num_plateaus = 2

# plateau_factor: clipping height relative to mean bin count.
#   Lower = stronger clipping · range 0.5–1.0 · default 0.7.
plateau_factor = 0.7

# Apply histogram equalization to boost contrast before thresholding.
im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=num_plateaus,
    plateau_factor=plateau_factor,
)

In [ ]:
#[Cell 15]
# ---------------------------------------------------------------------------
# Thresholding
# ---------------------------------------------------------------------------
# threshold_method: global algorithm used as one component of the combined
# threshold.
#   'otsu'   : maximises inter-class variance (default; bimodal histograms).
#   'median' : threshold at median non-zero intensity (sparse signal).
#   'huang'  : fuzzy entropy (dim or diffuse signal).
threshold_method = 'otsu'

# Apply combined thresholding (global + Sauvola local + statistical background).
# The global/Sauvola/statistical blend weights are fixed internally (not
# user-tunable) to keep thresholding behaviour consistent across datasets.
im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=threshold_method,
    progress=tqdm,
)

In [ ]:
#[Cell 16]
# Export histograms for every processing stage and a Parameters sheet.
output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    zoom_factors=zoom_factors, scale_factor=scale_factor,
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    multilabel=multilabel,
    nuclei_split_config=nuclei_split_config,
    sigma=sigma, num_plateaus=num_plateaus, plateau_factor=plateau_factor,
    threshold_method=threshold_method,
    progress=tqdm,
)

In [ ]:
#[Cell 17]
# Segment cells using Cellpose, StarDist, or watershed.
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    nuclei_split_config=nuclei_split_config,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    progress=tqdm,
)

# Add a virtual NUCLEI entry so downstream helpers can reference it.
if 'NUCLEI' not in stain_complete_df.index:
    stain_complete_df.loc['NUCLEI'] = ['', '', '', '', '', '']

In [ ]:
#[Cell 18]
# Assign segmented cell labels to each LIVE/DEAD channel.
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    progress=tqdm,
)

In [ ]:
#[Cell 19]
# Open napari viewers showing all processing stages and segmentation results.
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

This section measures properties of each segmented object and produces summary statistics and distribution plots.

- **Cell 20** — computes per-object marker overlap, intensity, volume, and centroid position. The resulting DataFrame is displayed for review.
- **Cell 21** — prints a population-level summary (LIVE/DEAD counts and percentages).
- **Cell 22** — builds the full (untruncated) quantification table at original resolution for export.
- **Cells 23-24** — plot spatial and size distributions of the LIVE/DEAD populations.

In [ ]:
#[Cell 20]
# Quantify marker overlap and intensity per segmented object.
labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    multilabel=multilabel,
    progress=tqdm,
)

In [ ]:
#[Cell 21]
# Print population-level summary: LIVE/DEAD cell counts and percentages.
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

In [ ]:
#[Cell 22]
# Build the full (untruncated) quantification table at original resolution for export.
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    progress=tqdm,
)

In [ ]:
#[Cell 23]
# Plot 2D and 3D spatial distributions of LIVE/DEAD cells.
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_final_stack=im_final_stack,
    r_X=r_X, r_Y=r_Y, r_Z=r_Z,
    zoom_factors=zoom_factors,
    progress=tqdm,
)

In [ ]:
#[Cell 24]
# Plot size distributions (volume, diameter) for each LIVE/DEAD population.
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

# 6. Export

This section writes all results to disk for downstream use.

- **Cell 25** — exports labelled VTK volume meshes for 3D visualization (e.g. in ParaView).
- **Cell 26** — exports per-marker binary STL meshes.
- **Cell 27** — writes the full quantification table, stain settings, and processing parameters to an Excel file.
- **Cell 28** — generates an Abaqus `.inp` mesh from the segmentation for finite element analysis.

In [ ]:
#[Cell 25]
# Build labelled VTK volume meshes for each segmented LIVE/DEAD population.
build_vtk_volumes(
    im_segmentation_stack,
    labels_full_df=labels_full_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)

In [ ]:
#[Cell 26]
# Export per-marker binary volumes as STL mesh files.
export_marker_stl(
    im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    progress=tqdm,
)

In [ ]:
#[Cell 27]
# Export full quantification results to Excel.
# The output file is saved as <input_file_stem>_segmentation.xlsx automatically.
output_path = export_quantification_to_excel(
    input_file,
    original_stain_complete_df,
    labels_full_df,
    progress=tqdm,
)
print(f"Saved to: {output_path}")

In [ ]:
#[Cell 28]
# Generate an Abaqus .inp mesh from the segmentation for finite element analysis.
export_fea_mesh(
    im_segmentation_stack,
    input_file=input_file,
    progress=tqdm,
)